# Exploración del featured RoCoLe

Objetivo: revisar la tabla de características (índices RGB y textura GLCM), chequear calidad, mirar outliers univariados, comparar sanas y afectadas en train, y documentar la decisión sobre VARI. No entrenamos modelos aquí.


Fijamos la semilla para que lo que corramos después sea reproducible.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path


In [ ]:
seed = 42

seed


Apuntamos a la tabla featured de RoCoLe (índices RGB y GLCM).


In [ ]:
repo_root = Path("../..").resolve()
features_path = repo_root / "data" / "featured" / "rocole" / "rgb_glcm_features.csv"
results_dir = repo_root / "experiments" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

print("features_path:", features_path)
print("existe:", features_path.exists())
features_path


Leemos la tabla: una fila por hoja, con etiqueta y medidas.


In [ ]:
leaf_features = pd.read_csv(features_path)

leaf_features.head()


## Inspección de la tabla

Miramos cuántas filas y columnas hay, y las primeras filas.


In [ ]:
# leaf_features

print("forma:", leaf_features.shape)
print("columnas:", list(leaf_features.columns))

leaf_features.head()


Revisamos si el mismo archivo aparece más de una vez. Si hay duplicados en File, hay que mirarlos antes de seguir.


In [ ]:
# leaf_features

duplicate_files = leaf_features.loc[leaf_features["File"].duplicated(keep=False)].copy()

print("filas con File repetido:", len(duplicate_files))
print("Files únicos duplicados:", duplicate_files["File"].nunique() if len(duplicate_files) else 0)
duplicate_files.head()


Apartamos las nueve medidas numéricas (ExG, NGRDI, VARI y GLCM). Las etiquetas quedan fuera de esta lista.


In [ ]:
# leaf_features

quality_columns = [
    "exg_mean",
    "exg_std",
    "ngrdi_mean",
    "ngrdi_std",
    "vari_mean",
    "vari_std",
    "glcm_contrast",
    "glcm_homogeneity",
    "glcm_energy",
]
measure_columns = quality_columns.copy()

quality_columns


Contamos nulos por columna en toda la tabla, con foco en las nueve medidas.


In [ ]:
null_counts = leaf_features.isna().sum()
missing_counts_all = leaf_features[quality_columns].isna().sum()

print("nulos en toda la tabla:")
print(null_counts)
print("\nnulos en las nueve medidas:")
print(missing_counts_all)
missing_counts_all


Contamos valores no finitos (infinitos o NaN) en las nueve medidas.


In [ ]:
# leaf_features, quality_columns

infinite_counts_all = {
    column: int((~np.isfinite(leaf_features[column])).sum())
    for column in quality_columns
}

print("no finitos por medida:")
for column, count in infinite_counts_all.items():
    print(f"  {column}: {count}")
infinite_counts_all


Miramos la distribución de ExG (exg_mean, exg_std) con diagramas de caja. Solo observamos; no cortamos outliers.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, column in zip(axes, ["exg_mean", "exg_std"]):
    sns.boxplot(y=leaf_features[column], ax=axis, color="steelblue")
    axis.set_title(column)
fig.suptitle("Cajas ExG")
fig.tight_layout()
exg_box_path = results_dir / "rocole-featured-exg-boxplots.png"
fig.savefig(exg_box_path, dpi=120, bbox_inches="tight")
plt.show()

print("figura:", exg_box_path)
exg_box_path


Misma mirada univariada de ExG en dispersión: cada punto es una hoja (eje x = índice de fila).


In [ ]:
# leaf_features

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
row_index = np.arange(len(leaf_features))
for axis, column in zip(axes, ["exg_mean", "exg_std"]):
    axis.scatter(row_index, leaf_features[column], s=8, alpha=0.5)
    axis.set_xlabel("índice de fila")
    axis.set_ylabel(column)
    axis.set_title(column)
fig.suptitle("Dispersión univariada ExG")
fig.tight_layout()
exg_scatter_path = results_dir / "rocole-featured-exg-scatter.png"
fig.savefig(exg_scatter_path, dpi=120, bbox_inches="tight")
plt.show()

print("figura:", exg_scatter_path)
exg_scatter_path


Ahora las cajas de NGRDI (ngrdi_mean, ngrdi_std), misma mirada univariada.


In [ ]:
# leaf_features

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, column in zip(axes, ["ngrdi_mean", "ngrdi_std"]):
    sns.boxplot(y=leaf_features[column], ax=axis, color="steelblue")
    axis.set_title(column)
fig.suptitle("Cajas NGRDI")
fig.tight_layout()
ngrdi_box_path = results_dir / "rocole-featured-ngrdi-boxplots.png"
fig.savefig(ngrdi_box_path, dpi=120, bbox_inches="tight")
plt.show()

print("figura:", ngrdi_box_path)
ngrdi_box_path


Dispersión univariada de NGRDI, una hoja por punto.


In [ ]:
# leaf_features

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, column in zip(axes, ["ngrdi_mean", "ngrdi_std"]):
    axis.scatter(row_index, leaf_features[column], s=8, alpha=0.5)
    axis.set_xlabel("índice de fila")
    axis.set_ylabel(column)
    axis.set_title(column)
fig.suptitle("Dispersión univariada NGRDI")
fig.tight_layout()
ngrdi_scatter_path = results_dir / "rocole-featured-ngrdi-scatter.png"
fig.savefig(ngrdi_scatter_path, dpi=120, bbox_inches="tight")
plt.show()

print("figura:", ngrdi_scatter_path)
ngrdi_scatter_path


Cajas de VARI (vari_mean, vari_std). Pueden verse aplastadas por extremos; aquí solo miramos, sin decidir todavía.


In [ ]:
# leaf_features

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, column in zip(axes, ["vari_mean", "vari_std"]):
    sns.boxplot(y=leaf_features[column], ax=axis, color="steelblue")
    axis.set_title(column)
fig.suptitle("Cajas VARI")
fig.tight_layout()
vari_box_path = results_dir / "rocole-featured-vari-boxplots.png"
fig.savefig(vari_box_path, dpi=120, bbox_inches="tight")
plt.show()

print("figura:", vari_box_path)
vari_box_path


Dispersión univariada de VARI. Los extremos saltan a la vista; seguimos observando.


In [ ]:
# leaf_features

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, column in zip(axes, ["vari_mean", "vari_std"]):
    axis.scatter(row_index, leaf_features[column], s=8, alpha=0.5)
    axis.set_xlabel("índice de fila")
    axis.set_ylabel(column)
    axis.set_title(column)
fig.suptitle("Dispersión univariada VARI")
fig.tight_layout()
vari_scatter_path = results_dir / "rocole-featured-vari-scatter.png"
fig.savefig(vari_scatter_path, dpi=120, bbox_inches="tight")
plt.show()

print("figura:", vari_scatter_path)
vari_scatter_path


Documentamos VARI en train con percentiles y extremos. La mediana suele verse razonable; las colas confirman el artefacto del denominador.


In [ ]:
# leaf_features, seed

train_leaves = leaf_features.loc[leaf_features["split"] == "train"].copy()
percentile_levels = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
vari_mean_percentiles = train_leaves["vari_mean"].quantile(percentile_levels)
vari_std_percentiles = train_leaves["vari_std"].quantile(percentile_levels)
vari_mean_extremes = {
    "min": float(train_leaves["vari_mean"].min()),
    "max": float(train_leaves["vari_mean"].max()),
    "min_file": train_leaves.loc[train_leaves["vari_mean"].idxmin(), "File"],
    "max_file": train_leaves.loc[train_leaves["vari_mean"].idxmax(), "File"],
}
vari_std_extremes = {
    "min": float(train_leaves["vari_std"].min()),
    "max": float(train_leaves["vari_std"].max()),
    "min_file": train_leaves.loc[train_leaves["vari_std"].idxmin(), "File"],
    "max_file": train_leaves.loc[train_leaves["vari_std"].idxmax(), "File"],
}

print("train filas:", len(train_leaves))
print("\nvari_mean percentiles:")
print(vari_mean_percentiles)
print("\nvari_mean extremos:", vari_mean_extremes)
print("\nvari_std percentiles:")
print(vari_std_percentiles)
print("\nvari_std extremos:", vari_std_extremes)
vari_mean_percentiles


Con esa evidencia dejamos documentada la decisión: retirar las columnas VARI del protocolo de modelado principal. El CSV las puede conservar; los modelos 04 y 05 no las usan.


In [ ]:
# evidencia de percentiles y figuras VARI

vari_feature_decision = "drop_vari"

print("vari_feature_decision:", vari_feature_decision)
vari_feature_decision


Cajas de las tres medidas GLCM: contraste, homogeneidad y energía.


In [ ]:
# leaf_features

glcm_columns = ["glcm_contrast", "glcm_homogeneity", "glcm_energy"]
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, column in zip(axes, glcm_columns):
    sns.boxplot(y=leaf_features[column], ax=axis, color="steelblue")
    axis.set_title(column)
fig.suptitle("Cajas GLCM")
fig.tight_layout()
glcm_box_path = results_dir / "rocole-featured-glcm-boxplots.png"
fig.savefig(glcm_box_path, dpi=120, bbox_inches="tight")
plt.show()

print("figura:", glcm_box_path)
glcm_box_path


Dispersión univariada de las medidas GLCM.


In [ ]:
# leaf_features

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for axis, column in zip(axes, glcm_columns):
    axis.scatter(row_index, leaf_features[column], s=8, alpha=0.5)
    axis.set_xlabel("índice de fila")
    axis.set_ylabel(column)
    axis.set_title(column)
fig.suptitle("Dispersión univariada GLCM")
fig.tight_layout()
glcm_scatter_path = results_dir / "rocole-featured-glcm-scatter.png"
fig.savefig(glcm_scatter_path, dpi=120, bbox_inches="tight")
plt.show()

print("figura:", glcm_scatter_path)
glcm_scatter_path


Hasta aquí: carga, duplicados, nulos, no finitos, mirada univariada y decisión de retirar VARI. Siguiente: comparar sanas y afectadas en train.


Comparamos hojas sanas y afectadas solo con train. VARI queda fuera de esta mirada.


Nos quedamos con las filas de aprender (split igual a train), si aún no estaban apartadas.


In [ ]:
# leaf_features

train_leaves = leaf_features.loc[leaf_features["split"] == "train"].copy()

print("train filas:", len(train_leaves))
train_leaves.head()


Contamos cuántas hojas de train hay por Binary.Label (sana frente a afectada).


In [ ]:
# train_leaves

train_label_counts = train_leaves["Binary.Label"].value_counts()

print(train_label_counts)
train_label_counts


Elegimos las columnas a comparar por clase: ExG, NGRDI y GLCM. Sin VARI.


In [ ]:
comparison_columns = [
    "exg_mean",
    "exg_std",
    "ngrdi_mean",
    "ngrdi_std",
    "glcm_contrast",
    "glcm_homogeneity",
    "glcm_energy",
]

comparison_columns


Miramos ExG (exg_mean, exg_std) con cajas lado a lado por Binary.Label, solo en train.


In [ ]:
# train_leaves

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, column in zip(axes, ["exg_mean", "exg_std"]):
    sns.boxplot(data=train_leaves, x="Binary.Label", y=column, ax=axis)
    axis.set_title(column)
fig.suptitle("ExG por Binary.Label (train)")
fig.tight_layout()
exg_by_label_path = results_dir / "rocole-featured-exg-by-label-boxplots.png"
fig.savefig(exg_by_label_path, dpi=120, bbox_inches="tight")
plt.show()

print("figura:", exg_by_label_path)
exg_by_label_path


Lectura ExG: mira la figura. Si las cajas se solapan mucho, las clases se mezclan en esa medida; si se apartan un poco, hay alguna diferencia visual. No es aún un veredicto de predicción.


Misma comparación con NGRDI (ngrdi_mean, ngrdi_std) por Binary.Label en train.


In [ ]:
# train_leaves

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, column in zip(axes, ["ngrdi_mean", "ngrdi_std"]):
    sns.boxplot(data=train_leaves, x="Binary.Label", y=column, ax=axis)
    axis.set_title(column)
fig.suptitle("NGRDI por Binary.Label (train)")
fig.tight_layout()
ngrdi_by_label_path = results_dir / "rocole-featured-ngrdi-by-label-boxplots.png"
fig.savefig(ngrdi_by_label_path, dpi=120, bbox_inches="tight")
plt.show()

print("figura:", ngrdi_by_label_path)
ngrdi_by_label_path


Lectura NGRDI: otra vez, mira el solape o la separación de las cajas. Es una lectura visual; no es un ranking de variables.


Ahora las tres medidas GLCM (contraste, homogeneidad, energía) por clase en train.


In [ ]:
# train_leaves

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for axis, column in zip(axes, ["glcm_contrast", "glcm_homogeneity", "glcm_energy"]):
    sns.boxplot(data=train_leaves, x="Binary.Label", y=column, ax=axis)
    axis.set_title(column)
fig.suptitle("GLCM por Binary.Label (train)")
fig.tight_layout()
glcm_by_label_path = results_dir / "rocole-featured-glcm-by-label-boxplots.png"
fig.savefig(glcm_by_label_path, dpi=120, bbox_inches="tight")
plt.show()

print("figura:", glcm_by_label_path)
glcm_by_label_path


Lectura GLCM: observa si alguna medida se aparta un poco entre clases o si casi todo se solapa. Seguimos sin elegir predictoras.


Vimos solape o separación a ojo entre sanas y afectadas en train (ExG, NGRDI, GLCM; sin VARI). Aún no elegimos variables ni medimos acierto.


Por último, comprobamos en train que etiqueta, split y link_group coinciden con el índice featured, fila a fila por File.


In [ ]:
# train_leaves

index_path = repo_root / "data" / "featured" / "rocole" / "index.csv"
featured_index = pd.read_csv(index_path)
train_with_index = train_leaves.merge(
    featured_index[["File", "Binary.Label", "split", "link_group"]],
    on="File",
    how="left",
    suffixes=("_feat", "_index"),
)
label_match = (
    train_with_index["Binary.Label_feat"] == train_with_index["Binary.Label_index"]
).all()
split_match = (train_with_index["split_feat"] == train_with_index["split_index"]).all()
link_match = (
    train_with_index["link_group_feat"] == train_with_index["link_group_index"]
).all()
missing_in_index = int(train_with_index["split_index"].isna().sum())
index_integrity = {
    "train_rows": len(train_leaves),
    "merged_rows": len(train_with_index),
    "missing_in_index": missing_in_index,
    "Binary.Label_match": bool(label_match),
    "split_match": bool(split_match),
    "link_group_match": bool(link_match),
}

print(index_integrity)
index_integrity


Antes de cerrar, miramos relaciones entre medidas y con la etiqueta. Solo hojas de aprender, sin VARI (ya quedó drop_vari). Las cajas ya se solapaban; aquí observamos, no elegimos predictoras fuertes.


Pasamos la etiqueta binaria a números para poder correlacionar: sana vale 0 y afectada vale 1.


In [ ]:
# train_leaves

binary_label_code = train_leaves["Binary.Label"].map(
    {"healthy": 0, "unhealthy": 1}
)

print(binary_label_code.value_counts(dropna=False))
binary_label_code


Armamos una tabla de train con las siete medidas ya usadas en la comparación por clase (ExG, NGRDI y GLCM) más el código 0/1. Sin columnas VARI.


In [ ]:
# train_leaves, comparison_columns, binary_label_code

if "comparison_columns" not in globals():
    comparison_columns = [
        "exg_mean",
        "exg_std",
        "ngrdi_mean",
        "ngrdi_std",
        "glcm_contrast",
        "glcm_homogeneity",
        "glcm_energy",
    ]

correlation_leaves = train_leaves[comparison_columns].copy()
correlation_leaves["binary_label_code"] = binary_label_code.to_numpy()

correlation_leaves.head()


Correlación de cada medida con el código de etiqueta. Miramos magnitud y signo; no es un ranking de variables predictoras.


In [ ]:
# correlation_leaves

feature_label_correlation = correlation_leaves[comparison_columns].corrwith(
    correlation_leaves["binary_label_code"]
)

print(feature_label_correlation)
feature_label_correlation


Lee esos números. Si salen bajos o cerca de cero, encaja con el solape de las cajas. No digas que una medida predice bien.


Ahora la relación entre medidas: matriz de correlación solo entre las siete características, sin la etiqueta.


In [ ]:
# correlation_leaves, comparison_columns

feature_correlation_matrix = correlation_leaves[comparison_columns].corr()

feature_correlation_matrix


Pintamos esa matriz como mapa de calor para ver qué medidas van juntas.


In [ ]:
# feature_correlation_matrix

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    feature_correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    ax=ax,
)
ax.set_title("Correlación entre medidas (train, sin VARI)")
fig.tight_layout()
feature_corr_heatmap_path = results_dir / "rocole-featured-feature-corr-heatmap.png"
fig.savefig(feature_corr_heatmap_path, dpi=120, bbox_inches="tight")
plt.show()

print("figura:", feature_corr_heatmap_path)
feature_corr_heatmap_path


Para no saturar la figura, elegimos unas pocas medidas de familias distintas: media de ExG, media de NGRDI y contraste GLCM.


In [ ]:
# comparison_columns

pair_feature_columns = ["exg_mean", "ngrdi_mean", "glcm_contrast"]

pair_feature_columns


Gráficos de pares solo con ese subconjunto, en train, coloreados por sana/afectada.


In [ ]:
# correlation_leaves, pair_feature_columns

pair_plot_data = correlation_leaves[pair_feature_columns].copy()
pair_plot_data["Binary.Label"] = correlation_leaves["binary_label_code"].map(
    {0: "healthy", 1: "unhealthy"}
)
pair_grid = sns.pairplot(
    pair_plot_data,
    vars=pair_feature_columns,
    hue="Binary.Label",
    diag_kind="hist",
)
pair_grid.fig.suptitle("Pares de medidas por Binary.Label (train)", y=1.02)
feature_pairs_path = results_dir / "rocole-featured-feature-pairs.png"
pair_grid.savefig(feature_pairs_path, dpi=120, bbox_inches="tight")
plt.show()

print("figura:", feature_pairs_path)
feature_pairs_path


Observa si las nubes de sana y afectada se mezclan. Seguimos sin coronar predictoras; el solape visual es el mensaje.


## Cierre

Checklist:
- Duplicados, nulos y no finitos revisados; las nueve medidas salen limpias en sentido de NaN/Inf.
- VARI queda documentada (percentiles y extremos) y se retira del modelado principal.
- En train, las cajas por clase se solapan en ExG, NGRDI y GLCM.
- Correlaciones y pares en train, sin VARI, muestran solapes o asociaciones débiles; no declaramos predictoras fuertes.
- Etiqueta, split y link_group coinciden con el índice por File.

Siguiente: cuadernos 04 y 05, sin columnas VARI.
